In [6]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os # Import the os module

# ---- CONFIG ----
CSV_PATH = "amazon_delivery.csv"
OUTPUT_DIR = "amazon_charts/"
MAX_REASONABLE_DISTANCE_KM = 100

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two lat/lon points."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def clean_data(df):
    """Clean and prepare the dataset."""
    df.columns = [c.strip() for c in df.columns]

    # Compute delivery distance
    df["distance_km"] = haversine(
        df["Store_Latitude"], df["Store_Longitude"],
        df["Drop_Latitude"], df["Drop_Longitude"]
    )

    # Extract hour of day
    df["order_hour"] = pd.to_datetime(
        df["Order_Time"], format="%H:%M:%S", errors="coerce"
    ).dt.hour

    # Clean numeric columns
    df['Agent_Rating'] = pd.to_numeric(df['Agent_Rating'], errors='coerce')
    df['Agent_Age'] = pd.to_numeric(df['Agent_Age'], errors='coerce')
    df['Delivery_Time'] = pd.to_numeric(df['Delivery_Time'], errors='coerce')

    # Clean categorical columns
    for col in ['Weather', 'Traffic', 'Vehicle', 'Area', 'Category']:
        if col in df.columns:
            df[col] = df[col].str.strip()

    # Filter out bad coordinates
    df_clean = df[df["distance_km"] <= MAX_REASONABLE_DISTANCE_KM]

    return df_clean, df



def main():
    # Load and clean data
    print("Loading data...")
    df_raw = pd.read_csv(CSV_PATH)
    df_clean, df = clean_data(df_raw)

    print(f"Total rows: {len(df_raw)}")
    print(f"Excluded bad coordinates: {len(df_raw) - len(df_clean)}")
    print(f"Cleaned rows: {len(df_clean)}")

    # Create output directory if it doesn't exist
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ======================================================================
    # PAIR 1: TIME-BASED ANALYSIS
    # ======================================================================
    print("\n📊 Generating Pair 1: Time-Based Analysis...")

    fig1, axes1 = plt.subplots(1, 2, figsize=(12, 4.8))

    # Panel 1: Order volume by hour
    hourly = df_clean["order_hour"].value_counts().sort_index()
    axes1[0].bar(hourly.index, hourly.values, color="#2c6e9e", width=0.7)
    axes1[0].set_title("Order Volume by Hour of Day", fontsize=12, fontweight="bold")
    axes1[0].set_xlabel("Hour of Day (24h)")
    axes1[0].set_ylabel("Number of Orders")
    axes1[0].set_xticks(range(0, 24, 2))
    axes1[0].grid(axis="y", alpha=0.3)

    # Mark peak hours
    peak_hours = hourly.nlargest(3)
    for peak in peak_hours.index:
        axes1[0].axvline(x=peak, color='red', linestyle='--', alpha=0.5, linewidth=1)
    axes1[0].text(0.02, 0.95, "Peak hours highlighted", transform=axes1[0].transAxes,
                  fontsize=9, color='red', alpha=0.7)

    # Panel 2: Distribution of delivery times
    axes1[1].hist(df_clean["Delivery_Time"], bins=30, color="#c76b3e", edgecolor="white", alpha=0.7)
    axes1[1].axvline(df_clean["Delivery_Time"].mean(), color="black", linestyle="--",
                     linewidth=1.5, label=f"Mean = {df_clean['Delivery_Time'].mean():.1f} min")
    axes1[1].axvline(df_clean["Delivery_Time"].median(), color="green", linestyle="--",
                     linewidth=1.5, label=f"Median = {df_clean['Delivery_Time'].median():.1f} min")
    axes1[1].set_title("Distribution of Delivery Times", fontsize=12, fontweight="bold")
    axes1[1].set_xlabel("Delivery Time (minutes)")
    axes1[1].set_ylabel("Number of Orders")
    axes1[1].legend()
    axes1[1].grid(axis="y", alpha=0.3)

    fig1.suptitle("Pair 1: Time-Based Analysis - Order Patterns & Delivery Duration",
                  fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}pair_1_time_analysis.png", dpi=200, bbox_inches="tight")
    plt.close()
    print("   ✅ Saved: pair_1_time_analysis.png")

    # ======================================================================
    # PAIR 2: GEOGRAPHIC ANALYSIS
    # ======================================================================
    print("\n📊 Generating Pair 2: Geographic Analysis...")

    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4.8))

    # Panel 1: Delivery distance distribution
    axes2[0].hist(df_clean["distance_km"], bins=40, color="#1f77b4", edgecolor="white", alpha=0.7)
    axes2[0].axvline(df_clean["distance_km"].mean(), color="black", linestyle="--",
                     linewidth=1.5, label=f"Mean = {df_clean['distance_km'].mean():.1f} km")
    axes2[0].axvline(df_clean["distance_km"].median(), color="green", linestyle="--",
                     linewidth=1.5, label=f"Median = {df_clean['distance_km'].median():.1f} km")
    axes2[0].set_title("Distribution of Delivery Distances", fontsize=12, fontweight="bold")
    axes2[0].set_xlabel("Store-to-Drop Distance (km)")
    axes2[0].set_ylabel("Number of Orders")
    axes2[0].legend()
    axes2[0].grid(axis="y", alpha=0.3)

    # Panel 2: Delivery time by area type
    area_order = df_clean.groupby('Area')['Delivery_Time'].median().sort_values().index
    sns.boxplot(data=df_clean, x='Area', y='Delivery_Time', order=area_order,
                ax=axes2[1], palette='Set2', showmeans=True,
                meanprops={"marker":"o", "markersize":5}, hue='Area', legend=False)
    axes2[1].set_title("Delivery Time by Area Type", fontsize=12, fontweight="bold")
    axes2[1].set_xlabel("Area Type")
    axes2[1].set_ylabel("Delivery Time (minutes)")
    axes2[1].grid(axis="y", alpha=0.3)

    fig2.suptitle("Pair 2: Geographic Analysis - Distances & Area Impact",
                  fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}pair_2_geographic_analysis.png", dpi=200, bbox_inches="tight")
    plt.close()
    print("   ✅ Saved: pair_2_geographic_analysis.png")

    # ======================================================================
    # PAIR 3: FACTOR ANALYSIS
    # ======================================================================
    print("\n📊 Generating Pair 3: Factor Analysis...")

    fig3, axes3 = plt.subplots(1, 2, figsize=(12, 4.8))

    # Panel 1: Delivery time by traffic condition
    traffic_order = df_clean.groupby('Traffic')['Delivery_Time'].median().sort_values().index
    sns.boxplot(data=df_clean, x='Traffic', y='Delivery_Time', order=traffic_order,
                ax=axes3[0], palette='Set3', showmeans=True,
                meanprops={"marker":"o", "markersize":5}, hue='Traffic', legend=False)
    axes3[0].set_title("Delivery Time by Traffic Condition", fontsize=12, fontweight="bold")
    axes3[0].set_xlabel("Traffic Condition")
    axes3[0].set_ylabel("Delivery Time (minutes)")
    axes3[0].grid(axis="y", alpha=0.3)

    # Panel 2: Delivery time by weather condition
    weather_order = df_clean.groupby('Weather')['Delivery_Time'].median().sort_values().index
    sns.boxplot(data=df_clean, x='Weather', y='Delivery_Time', order=weather_order,
                ax=axes3[1], palette='Set1', showmeans=True,
                meanprops={"marker":"o", "markersize":5}, hue='Weather', legend=False)
    axes3[1].set_title("Delivery Time by Weather Condition", fontsize=12, fontweight="bold")
    axes3[1].set_xlabel("Weather Condition")
    axes3[1].set_ylabel("Delivery Time (minutes)")
    axes3[1].grid(axis="y", alpha=0.3)

    fig3.suptitle("Pair 3: Factor Analysis - Traffic & Weather Impact",
                  fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}pair_3_factor_analysis.png", dpi=200, bbox_inches="tight")
    plt.close()
    print("   ✅ Saved: pair_3_factor_analysis.png")

    # ======================================================================
    # PAIR 4: AGENT PERFORMANCE
    # ======================================================================
    print("\n📊 Generating Pair 4: Agent Performance Analysis...")

    fig4, axes4 = plt.subplots(1, 2, figsize=(12, 4.8))

    # Panel 1: Agent Age vs Delivery Time
    df_age = df_clean.dropna(subset=['Agent_Age', 'Delivery_Time'])
    axes4[0].scatter(df_age['Agent_Age'], df_age['Delivery_Time'],
                     alpha=0.5, s=30, c='#1f77b4')

    # Add trend line
    if len(df_age) > 1:
        z = np.polyfit(df_age['Agent_Age'], df_age['Delivery_Time'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(df_age['Agent_Age'].min(), df_age['Agent_Age'].max(), 100)
        axes4[0].plot(x_line, p(x_line), color='red', linewidth=2, label='Trend')

    axes4[0].set_title("Agent Age vs Delivery Time", fontsize=12, fontweight="bold")
    axes4[0].set_xlabel("Agent Age (years)")
    axes4[0].set_ylabel("Delivery Time (minutes)")
    axes4[0].grid(alpha=0.3)
    axes4[0].legend()

    # Panel 2: Agent Rating vs Delivery Time
    df_rating = df_clean.dropna(subset=['Agent_Rating', 'Delivery_Time'])
    axes4[1].scatter(df_rating['Agent_Rating'], df_rating['Delivery_Time'],
                     alpha=0.5, s=30, c='#ff7f0e')

    # Add trend line
    if len(df_rating) > 1:
        z = np.polyfit(df_rating['Agent_Rating'], df_rating['Delivery_Time'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(df_rating['Agent_Rating'].min(), df_rating['Agent_Rating'].max(), 100)
        axes4[1].plot(x_line, p(x_line), color='red', linewidth=2, label='Trend')

    axes4[1].set_title("Agent Rating vs Delivery Time", fontsize=12, fontweight="bold")
    axes4[1].set_xlabel("Agent Rating")
    axes4[1].set_ylabel("Delivery Time (minutes)")
    axes4[1].grid(alpha=0.3)
    axes4[1].legend()

    fig4.suptitle("Pair 4: Agent Performance - Age & Rating Impact",
                  fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}pair_4_agent_performance.png", dpi=200, bbox_inches="tight")
    plt.close()
    print("   ✅ Saved: pair_4_agent_performance.png")

    # ======================================================================
    # PAIR 5: VEHICLE & CATEGORY ANALYSIS
    # ======================================================================
    print("\n📊 Generating Pair 5: Vehicle & Category Analysis...")

    fig5, axes5 = plt.subplots(1, 2, figsize=(12, 4.8))

    # Panel 1: Delivery time by vehicle type
    vehicle_order = df_clean.groupby('Vehicle')['Delivery_Time'].median().sort_values().index
    sns.violinplot(data=df_clean, x='Vehicle', y='Delivery_Time', order=vehicle_order,
                   ax=axes5[0], palette='coolwarm', inner='quartile', hue='Vehicle', legend=False)
    axes5[0].set_title("Delivery Time by Vehicle Type", fontsize=12, fontweight="bold")
    axes5[0].set_xlabel("Vehicle Type")
    axes5[0].set_ylabel("Delivery Time (minutes)")
    axes5[0].grid(axis="y", alpha=0.3)

    # Panel 2: Top categories by delivery time (horizontal bar)
    category_avg = df_clean.groupby('Category')['Delivery_Time'].mean().sort_values(ascending=True).tail(8)
    axes5[1].barh(category_avg.index, category_avg.values,
                  color=plt.cm.viridis(np.linspace(0.3, 0.9, len(category_avg))))
    axes5[1].set_title("Top 8 Categories by Average Delivery Time", fontsize=12, fontweight="bold")
    axes5[1].set_xlabel("Average Delivery Time (minutes)")
    axes5[1].set_ylabel("Category")
    axes5[1].grid(axis="x", alpha=0.3)

    fig5.suptitle("Pair 5: Operational Factors - Vehicle & Category Impact",
                  fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}pair_5_vehicle_category.png", dpi=200, bbox_inches="tight")
    plt.close()
    print("   ✅ Saved: pair_5_vehicle_category.png")

    # ======================================================================
    # COMPLETION SUMMARY
    # ======================================================================
    print("\n" + "="*70)
    print("✅ COMPLETE! All charts saved")
    print("="*70)
    print("\n📁 Files Created:")
    print("  📊 Charts:")
    print("   - pair_1_time_analysis.png")
    print("   - pair_2_geographic_analysis.png")
    print("   - pair_3_factor_analysis.png")
    print("   - pair_4_agent_performance.png")
    print("   - pair_5_vehicle_category.png")
    print("\n" + "="*70)


if __name__ == "__main__":
    main()

Loading data...
Total rows: 43739
Excluded bad coordinates: 188
Cleaned rows: 43551

📊 Generating Pair 1: Time-Based Analysis...
   ✅ Saved: pair_1_time_analysis.png

📊 Generating Pair 2: Geographic Analysis...
   ✅ Saved: pair_2_geographic_analysis.png

📊 Generating Pair 3: Factor Analysis...
   ✅ Saved: pair_3_factor_analysis.png

📊 Generating Pair 4: Agent Performance Analysis...
   ✅ Saved: pair_4_agent_performance.png

📊 Generating Pair 5: Vehicle & Category Analysis...
   ✅ Saved: pair_5_vehicle_category.png

✅ COMPLETE! All charts saved

📁 Files Created:
  📊 Charts:
   - pair_1_time_analysis.png
   - pair_2_geographic_analysis.png
   - pair_3_factor_analysis.png
   - pair_4_agent_performance.png
   - pair_5_vehicle_category.png

